In [6]:
## Run this cell to define Mamba2 module
# Copyright (c) 2024, Tri Dao, Albert Gu.

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat

try:
    from causal_conv1d import causal_conv1d_fn, causal_conv1d_update
except ImportError:
    causal_conv1d_fn, causal_conv1d_update = None, None

try:
    from causal_conv1d.causal_conv1d_varlen import causal_conv1d_varlen_states
except ImportError:
    causal_conv1d_varlen_states = None

try:
    from mamba_ssm.ops.triton.selective_state_update import selective_state_update
except ImportError:
    selective_state_update = None

from mamba_ssm.distributed.tensor_parallel import ColumnParallelLinear, RowParallelLinear
from mamba_ssm.distributed.distributed_utils import all_reduce, reduce_scatter

from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined
from mamba_ssm.ops.triton.ssd_combined import mamba_split_conv1d_scan_combined

from huggingface_hub import PyTorchModelHubMixin


class Mamba2(nn.Module, PyTorchModelHubMixin):
    def __init__(
        self,
        d_model,
        d_state=128,
        d_conv=4,
        conv_init=None,
        expand=2,
        headdim=64,
        d_ssm=None,  # If not None, we only apply SSM on this many dimensions, the rest uses gated MLP
        ngroups=1,
        A_init_range=(1, 16),
        D_has_hdim=False,
        rmsnorm=True,
        norm_before_gate=False,
        dt_min=0.001,
        dt_max=0.1,
        dt_init_floor=1e-4,
        dt_limit=(0.0, float("inf")),
        bias=False,
        conv_bias=True,
        # Fused kernel and sharding options
        chunk_size=256,
        use_mem_eff_path=True,
        layer_idx=None,  # Absorb kwarg for general module
        process_group=None,
        sequence_parallel=True,
        device=None,
        dtype=None,
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.conv_init = conv_init
        self.expand = expand
        self.process_group = process_group
        self.sequence_parallel = sequence_parallel
        self.world_size = 1 if process_group is None else process_group.size()
        self.local_rank = 0 if process_group is None else process_group.rank()
        self.d_inner = (self.expand * self.d_model) // self.world_size
        assert self.d_inner * self.world_size == self.expand * self.d_model
        self.headdim = headdim
        self.d_ssm = self.d_inner if d_ssm is None else d_ssm // self.world_size
        assert ngroups % self.world_size == 0
        self.ngroups = ngroups // self.world_size
        assert self.d_ssm % self.headdim == 0
        self.nheads = self.d_ssm // self.headdim
        self.D_has_hdim = D_has_hdim
        self.rmsnorm = rmsnorm
        self.norm_before_gate = norm_before_gate
        self.dt_limit = dt_limit
        self.activation = "silu"
        self.chunk_size = chunk_size
        self.use_mem_eff_path = use_mem_eff_path
        self.layer_idx = layer_idx

        # Order: [z, x, B, C, dt]
        d_in_proj = 2 * self.d_inner + 2 * self.ngroups * self.d_state + self.nheads
        if self.process_group is None:
            self.in_proj = nn.Linear(self.d_model, d_in_proj, bias=bias, **factory_kwargs)
        else:
            self.in_proj = ColumnParallelLinear(self.d_model, d_in_proj * self.world_size, bias=bias,
                                                process_group=self.process_group, sequence_parallel=self.sequence_parallel,
                                                **factory_kwargs)

        conv_dim = self.d_ssm + 2 * self.ngroups * self.d_state
        self.conv1d = nn.Conv1d(
            in_channels=conv_dim,
            out_channels=conv_dim,
            bias=conv_bias,
            kernel_size=d_conv,
            groups=conv_dim,
            padding=d_conv - 1,
            **factory_kwargs,
        )
        if self.conv_init is not None:
            nn.init.uniform_(self.conv1d.weight, -self.conv_init, self.conv_init)

        self.act = nn.GELU()

        # Initialize log dt bias
        dt = torch.exp(
            torch.rand(self.nheads, **factory_kwargs) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        dt = torch.clamp(dt, min=dt_init_floor)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)
        # Just to be explicit. Without this we already don't put wd on dt_bias because of the check
        # name.endswith("bias") in param_grouping.py
        self.dt_bias._no_weight_decay = True

        assert A_init_range[0] > 0 and A_init_range[1] >= A_init_range[0]
        A = torch.empty(self.nheads, dtype=torch.float32, device=device).uniform_(*A_init_range)
        A_log = torch.log(A).to(dtype=dtype)
        self.A_log = nn.Parameter(A_log)
        self.A_log._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.d_ssm if self.D_has_hdim else self.nheads, device=device))
        self.D._no_weight_decay = True

        if self.rmsnorm:
            # from torch.nn import SyncBatchNorm
            # self.norm = SyncBatchNorm(self.d_ssm)
            from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated
            assert RMSNormGated is not None
            self.norm = RMSNormGated(self.d_ssm, eps=1e-5, norm_before_gate=self.norm_before_gate,
                                     group_size=self.d_ssm // ngroups, **factory_kwargs)

        if self.process_group is None:
            self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias, **factory_kwargs)
        else:
            self.out_proj = RowParallelLinear(self.d_inner * self.world_size, self.d_model, bias=bias,
                                              process_group=self.process_group, sequence_parallel=self.sequence_parallel,
                                              **factory_kwargs)

    def forward(self, u, seqlen=None, seq_idx=None, cu_seqlens=None, inference_params=None):
        """
        u: (batch, seqlen, hidden_dim) if seqlen=None.
            If seqlen is not None, u is (batch * seqlen, hidden_dim). This is so that when we
            split u during sequence parallel, we split the batch * seqlen dimension
            (in case batch is small).
        Returns: same shape as u
        """
        seqlen_og = seqlen
        if seqlen is None:
            batch, seqlen, dim = u.shape
        else:
            batch_seqlen, dim = u.shape
            batch = batch_seqlen // seqlen

        conv_state, ssm_state = None, None
        if inference_params is not None:
            inference_batch = cu_seqlens.shape[0] - 1 if cu_seqlens is not None else batch
            conv_state, ssm_state = self._get_states_from_cache(inference_params, inference_batch)
            if inference_params.seqlen_offset > 0:
                # The states are updated inplace
                out, _, _ = self.step(u, conv_state, ssm_state)
                return out

        zxbcdt = self.in_proj(u)  # (B, L, d_in_proj) or (B * L, d_in_proj)
        if seqlen_og is not None:
            zxbcdt = rearrange(zxbcdt, "(b l) d -> b l d", l=seqlen)
        # If the model is loaded in fp16, without the .float() here, A might be -inf
        A = -torch.exp(self.A_log.float())  # (nheads) or (d_inner, d_state)
        dt_limit_kwargs = {} if self.dt_limit == (0.0, float("inf")) else dict(dt_limit=self.dt_limit)
        if self.use_mem_eff_path and inference_params is None:
            out = mamba_split_conv1d_scan_combined(
                zxbcdt,
                rearrange(self.conv1d.weight, "d 1 w -> d w"),
                self.conv1d.bias,
                self.dt_bias,
                A,
                D=rearrange(self.D, "(h p) -> h p", p=self.headdim) if self.D_has_hdim else self.D,
                chunk_size=self.chunk_size,
                seq_idx=seq_idx,
                activation=self.activation,
                rmsnorm_weight=self.norm.weight if self.rmsnorm else None,
                rmsnorm_eps=self.norm.eps if self.rmsnorm else 1e-6,
                outproj_weight=self.out_proj.weight,
                outproj_bias=self.out_proj.bias,
                headdim=None if self.D_has_hdim else self.headdim,
                ngroups=self.ngroups,
                norm_before_gate=self.norm_before_gate,
                **dt_limit_kwargs,
            )
            if seqlen_og is not None:
                out = rearrange(out, "b l d -> (b l) d")
            if self.process_group is not None:
                reduce_fn = reduce_scatter if self.sequence_parallel else all_reduce
                out = reduce_fn(out, self.process_group)
        else:
            d_mlp = (zxbcdt.shape[-1] - 2 * self.d_ssm - 2 * self.ngroups * self.d_state - self.nheads) // 2
            z0, x0, z, xBC, dt = torch.split(
                zxbcdt,
                [d_mlp, d_mlp, self.d_ssm, self.d_ssm + 2 * self.ngroups * self.d_state, self.nheads],
                dim=-1
            )
            if conv_state is not None:
                if cu_seqlens is None:
                    # If we just take xBC[:, :, -self.d_conv :], it will error if seqlen < self.d_conv
                    # Instead F.pad will pad with zeros if seqlen < self.d_conv, and truncate otherwise.
                    xBC_t = rearrange(xBC, "b l d -> b d l")
                    conv_state.copy_(F.pad(xBC_t, (self.d_conv - xBC_t.shape[-1], 0)))  # Update state (B D W)
                else:
                    assert causal_conv1d_varlen_states is not None, "varlen inference requires causal_conv1d package"
                    assert batch == 1, "varlen inference only supports batch dimension 1"
                    conv_varlen_states = causal_conv1d_varlen_states(
                        xBC.squeeze(0), cu_seqlens, state_len=conv_state.shape[-1]
                    )
                    conv_state.copy_(conv_varlen_states)
            assert self.activation in ["silu", "swish"]
            if causal_conv1d_fn is None or self.activation not in ["silu", "swish"]:
                assert seq_idx is None, "varlen conv1d requires the causal_conv1d package"
                xBC = self.act(
                    self.conv1d(xBC.transpose(1, 2)).transpose(1, 2)[:, :-(self.d_conv - 1)]
                )  # (B, L, self.d_ssm + 2 * ngroups * d_state)
            else:
                xBC = causal_conv1d_fn(
                    xBC.transpose(1, 2),
                    rearrange(self.conv1d.weight, "d 1 w -> d w"),
                    bias=self.conv1d.bias,
                    activation=self.activation,
                    seq_idx=seq_idx,
                ).transpose(1, 2)
            x, B, C = torch.split(xBC, [self.d_ssm, self.ngroups * self.d_state, self.ngroups * self.d_state], dim=-1)
            y = mamba_chunk_scan_combined(
                rearrange(x, "b l (h p) -> b l h p", p=self.headdim),
                dt,
                A,
                rearrange(B, "b l (g n) -> b l g n", g=self.ngroups),
                rearrange(C, "b l (g n) -> b l g n", g=self.ngroups),
                chunk_size=self.chunk_size,
                D=rearrange(self.D, "(h p) -> h p", p=self.headdim) if self.D_has_hdim else self.D,
                z=rearrange(z, "b l (h p) -> b l h p", p=self.headdim) if not self.rmsnorm else None,
                dt_bias=self.dt_bias,
                dt_softplus=True,
                seq_idx=seq_idx,
                cu_seqlens=cu_seqlens,
                **dt_limit_kwargs,
                return_final_states=ssm_state is not None,
                return_varlen_states=cu_seqlens is not None and inference_params is not None,
            )
            if ssm_state is not None:
                y, last_state, *rest = y
                if cu_seqlens is None:
                    ssm_state.copy_(last_state)
                else:
                    varlen_states = rest[0]
                    ssm_state.copy_(varlen_states)
            y = rearrange(y, "b l h p -> b l (h p)")
            if self.rmsnorm:
                y = self.norm(y, z)
            if d_mlp > 0:
                y = torch.cat([F.silu(z0) * x0, y], dim=-1)
            if seqlen_og is not None:
                y = rearrange(y, "b l d -> (b l) d")
            out = self.out_proj(y)
        return out

    def step(self, hidden_states, conv_state, ssm_state):
        dtype = hidden_states.dtype
        assert hidden_states.shape[1] == 1, "Only support decoding with 1 token at a time for now"
        zxbcdt = self.in_proj(hidden_states.squeeze(1))  # (B 2D)
        d_mlp = (zxbcdt.shape[-1] - 2 * self.d_ssm - 2 * self.ngroups * self.d_state - self.nheads) // 2
        z0, x0, z, xBC, dt = torch.split(
            zxbcdt,
            [d_mlp, d_mlp, self.d_ssm, self.d_ssm + 2 * self.ngroups * self.d_state, self.nheads],
            dim=-1
        )

        # Conv step
        if causal_conv1d_update is None:
            conv_state.copy_(torch.roll(conv_state, shifts=-1, dims=-1))  # Update state (B D W)
            conv_state[:, :, -1] = xBC
            xBC = torch.sum(conv_state * rearrange(self.conv1d.weight, "d 1 w -> d w"), dim=-1)  # (B D)
            if self.conv1d.bias is not None:
                xBC = xBC + self.conv1d.bias
            xBC = self.act(xBC).to(dtype=dtype)
        else:
            xBC = causal_conv1d_update(
                xBC,
                conv_state,
                rearrange(self.conv1d.weight, "d 1 w -> d w"),
                self.conv1d.bias,
                self.activation,
            )

        x, B, C = torch.split(xBC, [self.d_ssm, self.ngroups * self.d_state, self.ngroups * self.d_state], dim=-1)
        A = -torch.exp(self.A_log.float())  # (nheads,)

        # SSM step
        if selective_state_update is None:
            assert self.ngroups == 1, "Only support ngroups=1 for this inference code path"
            # Discretize A and B
            dt = F.softplus(dt + self.dt_bias.to(dtype=dt.dtype))  # (batch, nheads)
            dA = torch.exp(dt * A)  # (batch, nheads)
            x = rearrange(x, "b (h p) -> b h p", p=self.headdim)
            dBx = torch.einsum("bh,bn,bhp->bhpn", dt, B, x)
            ssm_state.copy_(ssm_state * rearrange(dA, "b h -> b h 1 1") + dBx)
            y = torch.einsum("bhpn,bn->bhp", ssm_state.to(dtype), C)
            y = y + rearrange(self.D.to(dtype), "h -> h 1") * x
            y = rearrange(y, "b h p -> b (h p)")
            if not self.rmsnorm:
                y = y * self.act(z)  # (B D)
        else:
            A = repeat(A, "h -> h p n", p=self.headdim, n=self.d_state).to(dtype=torch.float32)
            dt = repeat(dt, "b h -> b h p", p=self.headdim)
            dt_bias = repeat(self.dt_bias, "h -> h p", p=self.headdim)
            D = repeat(self.D, "h -> h p", p=self.headdim)
            B = rearrange(B, "b (g n) -> b g n", g=self.ngroups)
            C = rearrange(C, "b (g n) -> b g n", g=self.ngroups)
            x_reshaped = rearrange(x, "b (h p) -> b h p", p=self.headdim)
            if not self.rmsnorm:
                z = rearrange(z, "b (h p) -> b h p", p=self.headdim)
            y = selective_state_update(
                ssm_state, x_reshaped, dt, A, B, C, D, z=z if not self.rmsnorm else None,
                dt_bias=dt_bias, dt_softplus=True
            )
            y = rearrange(y, "b h p -> b (h p)")
        if self.rmsnorm:
            y = self.norm(y, z)
        if d_mlp > 0:
            y = torch.cat([F.silu(z0) * x0, y], dim=-1)
        out = self.out_proj(y)
        return out.unsqueeze(1), conv_state, ssm_state

    def allocate_inference_cache(self, batch_size, max_seqlen, dtype=None, **kwargs):
        device = self.out_proj.weight.device
        conv_dtype = self.conv1d.weight.dtype if dtype is None else dtype
        conv_state = torch.zeros(
            batch_size, self.d_conv, self.conv1d.weight.shape[0], device=device, dtype=conv_dtype
        ).transpose(1, 2)
        ssm_dtype = self.in_proj.weight.dtype if dtype is None else dtype
        ssm_state = torch.zeros(
            batch_size, self.nheads, self.headdim, self.d_state, device=device, dtype=ssm_dtype
        )
        return conv_state, ssm_state

    def _get_states_from_cache(self, inference_params, batch_size, initialize_states=False):
        assert self.layer_idx is not None
        if self.layer_idx not in inference_params.key_value_memory_dict:
            batch_shape = (batch_size,)
            conv_state = torch.zeros(
                batch_size,
                self.d_conv,
                self.conv1d.weight.shape[0],
                device=self.conv1d.weight.device,
                dtype=self.conv1d.weight.dtype,
            ).transpose(1, 2)
            ssm_state = torch.zeros(
                batch_size,
                self.nheads,
                self.headdim,
                self.d_state,
                device=self.in_proj.weight.device,
                dtype=self.in_proj.weight.dtype,
            )
            inference_params.key_value_memory_dict[self.layer_idx] = (conv_state, ssm_state)
        else:
            conv_state, ssm_state = inference_params.key_value_memory_dict[self.layer_idx]
            # TODO: What if batch size changes between generation, and we reuse the same states?
            if initialize_states:
                conv_state.zero_()
                ssm_state.zero_()
        return conv_state, ssm_state

In [7]:
## Run this cell to define GPSConv module

import inspect
from typing import Any, Dict, Optional

import torch
import torch.nn.functional as F
from torch import Tensor
from torch.nn import Dropout, Linear, Sequential

from torch.nn import Embedding, Linear, BatchNorm1d, ReLU, Sequential, SyncBatchNorm, GELU, SiLU

from torch_geometric.nn.attention import PerformerAttention
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import reset
from torch_geometric.nn.resolver import (
    activation_resolver,
    normalization_resolver,
)
from torch_geometric.typing import Adj
from torch_geometric.utils import to_dense_batch
from torch_geometric.utils import degree, sort_edge_index

class GPSConv(torch.nn.Module):
    r"""The general, powerful, scalable (GPS) graph transformer layer from the
    `"Recipe for a General, Powerful, Scalable Graph Transformer"
    <https://arxiv.org/abs/2205.12454>`_ paper.

    The GPS layer is based on a 3-part recipe:

    1. Inclusion of positional (PE) and structural encodings (SE) to the input
       features (done in a pre-processing step via
       :class:`torch_geometric.transforms`).
    2. A local message passing layer (MPNN) that operates on the input graph.
    3. A global attention layer that operates on the entire graph.

    .. note::

        For an example of using :class:`GPSConv`, see
        `examples/graph_gps.py
        <https://github.com/pyg-team/pytorch_geometric/blob/master/examples/
        graph_gps.py>`_.

    Args:
        channels (int): Size of each input sample.
        conv (MessagePassing, optional): The local message passing layer.
        heads (int, optional): Number of multi-head-attentions.
            (default: :obj:`1`)
        dropout (float, optional): Dropout probability of intermediate
            embeddings. (default: :obj:`0.`)
        act (str or Callable, optional): The non-linear activation function to
            use. (default: :obj:`"relu"`)
        act_kwargs (Dict[str, Any], optional): Arguments passed to the
            respective activation function defined by :obj:`act`.
            (default: :obj:`None`)
        norm (str or Callable, optional): The normalization function to
            use. (default: :obj:`"batch_norm"`)
        norm_kwargs (Dict[str, Any], optional): Arguments passed to the
            respective normalization function defined by :obj:`norm`.
            (default: :obj:`None`)
        attn_type (str): Global attention type, :obj:`multihead` or
            :obj:`performer`. (default: :obj:`multihead`)
        attn_kwargs (Dict[str, Any], optional): Arguments passed to the
            attention layer. (default: :obj:`None`)
    """
    def __init__(
        self,
        channels: int,
        heads: int = 1,
        dropout: float = 0.0,
        act: str = 'relu',
        act_kwargs: Optional[Dict[str, Any]] = None,
        norm: Optional[str] = 'batch_norm',
        norm_kwargs: Optional[Dict[str, Any]] = None,
        attn_type: str = 'multihead',
        attn_kwargs: Optional[Dict[str, Any]] = None,
        ## Mamba parameters
        d_state: int = 512,
        d_conv: int = 3,
        expand: int = 4,
        headdim: int = 64,
        shuffle_ind: int = 0,
    ):
        super().__init__()

        self.channels = channels
        self.heads = heads
        self.dropout = dropout
        self.attn_type = attn_type
        self.shuffle_ind = shuffle_ind

        attn_kwargs = attn_kwargs or {}
        if attn_type == 'multihead':
            self.attn = torch.nn.MultiheadAttention(
                channels,
                heads,
                batch_first=True,
                **attn_kwargs,
            )
        elif attn_type == 'mamba2':
            self.attn = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
            # self.attn.norm = SyncBatchNorm(channels)
        elif attn_type == 'mamba2_bidirectional':
            # Mamba2 with bidirectional attention
            self.attn = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
            # self.attn.norm = SyncBatchNorm(channels)
            # Second Mamba2 for reverse scan
            # This is used to capture bidirectional dependencies
            self.attn2 = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
            # self.attn2.norm = SyncBatchNorm(channels)

        elif attn_type == 'mamba2_bidirectional_diff':
            self.learnable_lambda = torch.nn.Parameter(torch.tensor(0.0))
            # Mamba2 with bidirectional attention
            self.attn = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
            self.attn2 = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
            self.attn3 = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
            self.attn4 = Mamba2(
                d_model=channels,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=headdim,
            )
        else:
            raise ValueError(f'{attn_type} is not supported')

        self.mlp = Sequential(
            Linear(channels, channels * 2),
            GELU(),
            Dropout(dropout),
            Linear(channels * 2, channels),
            Dropout(dropout),
        )
        self.activation = GELU()

        norm_kwargs = norm_kwargs or {}
        self.norm1 = SyncBatchNorm(channels)
        # self.norm2 = SyncBatchNorm(channels)
        # self.norm3 = SyncBatchNorm(channels)

        self.norm_with_batch = False
        # if self.norm1 is not None:
        #     signature = inspect.signature(self.norm1.forward)
        #     self.norm_with_batch = 'batch' in signature.parameters

        # print(self.attn.norm.shape)

    def reset_parameters(self):
        r"""Resets all learnable parameters of the module."""
        if self.conv is not None:
            self.conv.reset_parameters()
        self.attn._reset_parameters()
        reset(self.mlp)
        if self.norm1 is not None:
            self.norm1.reset_parameters()
        if self.norm2 is not None:
            self.norm2.reset_parameters()
        if self.norm3 is not None:
            self.norm3.reset_parameters()

    def forward(
        self,
        x: Tensor,
        edge_index: Adj,
        batch: Optional[torch.Tensor] = None,
        batch_diff: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> Tensor:
        r"""Runs the forward pass of the module."""
        if isinstance(self.attn, torch.nn.MultiheadAttention):
            h, mask = to_dense_batch(h, batch)
            h, _ = self.attn(h, h, h, key_padding_mask=~mask, need_weights=False)
            h = h[mask]
        elif isinstance(self.attn, Mamba2):
            if self.attn_type == "mamba_bidirectional" or self.attn_type == "mamba2_bidirectional":
                ## forward scan
                h_forward, mask = to_dense_batch(x, batch)
                h_forward = self.attn(h_forward)
                h_forward = h_forward[mask]

                ## backward scan
                x_reverse = x.flip(0)
                batch_reverse = batch.flip(0)
                batch_reverse = torch.max(batch) - batch_reverse
                h_reverse, mask_reverse = to_dense_batch(x_reverse, batch_reverse)
                h_reverse = self.attn2(h_reverse)
                h_reverse = h_reverse[mask_reverse]
                h_reverse = h_reverse.flip(0)
                h = h_forward + h_reverse

            elif self.attn_type == "mamba2_bidirectional_diff":
                ## forward scan
                h_dense, mask = to_dense_batch(x, batch)
                h_forward = self.attn(h_dense)
                h_forward = h_forward[mask]

                ## backward scan
                x_reverse = x.flip(0)
                batch_reverse = batch.flip(0)
                batch_reverse = torch.max(batch) - batch_reverse
                h_dense_reverse, mask_reverse = to_dense_batch(x_reverse, batch_reverse)
                h_reverse = self.attn2(h_dense_reverse)
                h_reverse = h_reverse[mask_reverse]
                h_reverse = h_reverse.flip(0)

                ## forward scan
                h_forward_diff, mask_diff = to_dense_batch(x, batch_diff)
                h_forward_diff = self.attn3(h_forward_diff)
                h_forward_diff = h_forward_diff[mask_diff]

                ## backward scan
                batch_diff_reverse = batch_diff.flip(0)
                batch_diff_reverse = torch.max(batch_diff) - batch_diff_reverse
                h_diff_reverse, mask_diff_reverse = to_dense_batch(x_reverse, batch_diff_reverse)
                h_diff_reverse = self.attn4(h_diff_reverse)
                h_diff_reverse = h_diff_reverse[mask_diff_reverse]
                h_diff_reverse = h_diff_reverse.flip(0)
                h = h_forward + h_reverse - self.learnable_lambda * (h_forward_diff + h_diff_reverse)

            else:
                h, mask = to_dense_batch(x, batch)
                h = self.attn(h)[mask]

        if self.norm1 is not None:
            if self.norm_with_batch:
                h = self.norm1(h, batch=batch)
            else:
                h = self.norm1(h)
        h = self.activation(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.mlp(h)
        return h + x

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}({self.channels}, '
                f'conv={self.conv}, heads={self.heads}, '
                f'attn_type={self.attn_type})')

In [8]:
## Run this cell to define the full GNN model

import torch
import torch.nn as nn
import torch_geometric
import torch_geometric.nn as geom_nn
from torch.nn import Embedding, Linear, BatchNorm1d, ReLU, Sequential, SyncBatchNorm, GELU, SiLU


class GNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dp_rate):
        super().__init__()
        self.conv = geom_nn.GatedGraphConv(out_channels=out_channels, num_layers=2, aggr="max") # for GatedGraphConv
        self.bn = nn.SyncBatchNorm(out_channels) 
        self.activation = nn.GELU()
        # self.dropout = nn.Dropout(dp_rate)
        self.FFN =   Sequential(
            Linear(out_channels, out_channels * 4),
            GELU(),
            Dropout(dp_rate),
            Linear(out_channels * 4, out_channels),
            Dropout(dp_rate),
        )
    
    def forward(self, x, edge_index, batch=None):
        x_input = x
        x = self.conv(x, edge_index)
        x = self.bn(x)
        x = self.activation(x)
       #  x = self.dropout(x) 
        x = self.FFN(x) 
        return x + x_input


class GNNModel(nn.Module):
    
    def __init__(self, cfg):
        super().__init__()
        ## parameters
        attention_type = cfg.MODEL.attention_type
        c_in, c_hidden, c_out, num_layers, dp_rate = cfg.MODEL.c_in, cfg.MODEL.c_hidden, cfg.MODEL.c_out, cfg.MODEL.num_layers, cfg.MODEL.dp_rate
        d_state, d_conv, expand, headdim = cfg.MODEL.d_state, cfg.MODEL.d_conv, cfg.MODEL.expand, cfg.MODEL.headdim

        ## embedding
        self.node_emb = Linear(c_in, c_hidden)  # embedding for node features
        layers_gcn = []
        layers_global_mamba = []
        for i in range(num_layers-1):
            layers_gcn += [GNNBlock(in_channels=c_hidden, out_channels=c_hidden, dp_rate=dp_rate)]
            layers_global_mamba += [GPSConv(channels=c_hidden, attn_type=attention_type, d_state=d_state, d_conv=d_conv, expand=expand, dropout=dp_rate, shuffle_ind=1, headdim=headdim)]
        self.layers_gcn = nn.ModuleList(layers_gcn)
        self.layers_global_mamba = nn.ModuleList(layers_global_mamba)
        self.mlp = Linear(c_hidden, c_out)

    def get_batch(self, batch):
        x, edge_index = batch["x"], batch["edge_index"]
        batch_mask = batch["batch_mask"]
        output_mask = batch["output_mask"]
        return batch_mask, x, edge_index, output_mask

    def forward_cone(self, batch_mask, x, edge_index, batch_depth_mask=None):
        ## go through layers
        for i in range(len(self.layers_gcn)):
            x = self.layers_gcn[i](x, edge_index)
            x = self.layers_global_mamba[i](x, edge_index, batch=batch_mask, batch_diff=batch_depth_mask)
        return x
    
    def forward(self, batch):
        batch_mask, x_original, edge_index, output_mask = self.get_batch(batch)
        batch_depth_mask = batch["batch_depth_mask"]
        x_original = self.node_emb(x_original)
        x_cone = self.forward_cone(batch_mask, x_original, edge_index, batch_depth_mask)
        x_out = self.mlp(x_cone)
        x_out = x_out[output_mask, :]
        return x_out



In [13]:
from omegaconf import OmegaConf

cfg = OmegaConf.load("./configs/Sequencer.yaml")
cfg.MODEL.c_in = 21
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = GNNModel(cfg).to(device)


x = torch.rand((16,20)).to(device)
edge_index = torch.tensor([[0, 0, 0, 0, 1, 1, 1, 1,
                            8, 8, 8, 8, 9, 9, 9, 9],
                           [1, 2, 3, 4, 0, 2, 3, 4,
                            9,10,11,12, 8,10,11,12]], dtype=torch.long).to(device)
output_mask = torch.tensor([True, False, False, False, False, False, False, False,
                            True, False, False, False, False, False, False, False]).to(device)
batch_mask = torch.tensor((0, 0, 0, 0, 0, 0, 0, 0, 
                      1, 1, 1, 1, 1, 1, 1, 1)).to(device)
depth = torch.tensor((2, 1, 1, 1, 1, 0, 0, 0, 
                      2, 1, 1, 1, 1, 0, 0, 0)).to(device)
children_orders = torch.tensor((3, 0, 0, 1, 2, -1, -1, -1, 
                                3, 0, 0, 1, 2, -1, -1, -1,)).to(device)
## for additional mask
if "diff" in cfg.MODEL.attention_type:
    batch_depth_mask = torch.zeros(x.size(0), dtype=torch.long, device=device)
    batch_depth_mask_count = 0
    unique_batches = torch.unique(batch_mask)
    unique_depths = torch.unique(depth)
    for b in unique_batches:
        for d in unique_depths.flip(0):
            mask = (batch_mask == b) & (depth == d)
            if mask.sum() > 0:
                batch_depth_mask[mask] = batch_depth_mask_count
                batch_depth_mask_count += 1
    batch_depth_mask = batch_depth_mask.to(device)
else:
    batch_depth_mask = None

x = torch.cat([x, children_orders.unsqueeze(-1).float()], dim=-1).to(device)  # add children orders as feature
batch = {
    "x": x,
    "edge_index": edge_index,
    "output_mask": output_mask,
    "batch_mask": batch_mask,
    "batch_depth_mask": batch_depth_mask,
}
out = model(batch)
print(out.shape)  # should be (batch_size, c_out)

torch.Size([2, 1])
